# AutoGaze External CUDA Verification

This notebook verifies the pushed AutoGaze reproduction branch on a CUDA runtime. It is intended for Kaggle or Colab after GPU access is enabled.


In [ ]:
import json, os, pathlib, subprocess, sys, textwrap, time
import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. Enable a Kaggle/Colab GPU runtime before continuing.')
print('cuda_device:', torch.cuda.get_device_name(0))


In [ ]:
BRANCH = 'codex/autogaze-vjepa'
REPO_URL = 'https://github.com/manricheon/AutoGaze.git'
WORK_ROOT = pathlib.Path('/kaggle/working')
OUTPUT_ROOT = pathlib.Path('/kaggle/working/autogaze_vjepa_outputs')
WEIGHTS_ROOT = pathlib.Path('/kaggle/working/autogaze_weights')
REPORT_PATH = pathlib.Path('/kaggle/working/autogaze_vjepa_outputs/colab_verification.md')
REPO_DIR = WORK_ROOT / 'AutoGaze'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print('\n$', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)
os.chdir(REPO_DIR)
run(['git', 'log', '--oneline', '-1'])


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-repro.txt', 'transformers>=4.57.0', 'qwen-vl-utils', 'av', 'pytest'])


In [ ]:
run([
    sys.executable, 'scripts/verify_autogaze_entrypoints.py',
    '--output-json', OUTPUT_ROOT / 'entrypoint_verification.json',
    '--output-md', OUTPUT_ROOT / 'entrypoint_verification.md',
])


In [ ]:
run([
    sys.executable, 'scripts/run_colab_autogaze_cuda_smoke.py',
    '--weights-root', WEIGHTS_ROOT,
    '--output-root', OUTPUT_ROOT,
    '--video', 'inputs/hlvid_example/clip_av_video_5_001.mp4',
    '--num-video-frames', '16',
    '--frames-per-clip', '16',
    '--video-resize-longest-edge', '224',
    '--max-new-tokens', '4',
])


In [ ]:
run([
    sys.executable, '-m', 'repro.colab_verification_report',
    '--output-md', REPORT_PATH,
    '--title', 'AutoGaze External CUDA Verification',
    '--video', 'inputs/hlvid_example/clip_av_video_5_001.mp4',
    '--query', 'Describe the video in one short sentence.',
    '--entrypoint-verification-json', OUTPUT_ROOT / 'entrypoint_verification.json',
    '--case', f"vjepa_qwen_dense_off={OUTPUT_ROOT / 'vjepa_qwen_dense_off_cuda_smoke.json'}",
    '--case', f"autogaze_vjepa_qwen_on={OUTPUT_ROOT / 'autogaze_vjepa_qwen_on_cuda_smoke.json'}",
])


In [ ]:
summary_path = OUTPUT_ROOT / 'colab_autogaze_cuda_smoke_summary.json'
report_path = REPORT_PATH
summary = json.loads(summary_path.read_text())
print(json.dumps(summary['summary'], indent=2))
print('report:', report_path)
print('visualizations:', OUTPUT_ROOT / 'visualizations')
assert summary['summary']['passed'] is True
dense = summary['results']['vjepa_qwen_dense_off']
ag = summary['results']['autogaze_vjepa_qwen_on']
assert dense['status'] == 'passed'
assert ag['status'] == 'passed'
assert ag['tokens']['vjepa_selected_tokens'] < ag['tokens']['vjepa_raw_tokens']
assert ag['tokens']['qwen_visual_tokens_inserted'] == ag['tokens']['vjepa_selected_tokens']
